# NRGS in other LMs — combined figures (App. "Checking if NRGS Holds in Other LMs")

One figure per experiment, one row per LM, same colour scheme as the Coder-32B figures:
- `tvp_grids_by_lm` — task span vs code-progress span: residual transfer (Fig 27 right) + KV on 250 goal heads, V×1/2/4 (Fig 28 (i)).
- `kv_barcharts_by_lm` — pivot-onwards KV transfer of the task span (Fig 22).
- `q_barcharts_by_lm` — query transfer with gaze-select-restrict (Fig 26).

Other LMs' NRGS readouts use the replace strategy (α 1, layers 35-45; arm suffix `__ros-replace-a1-L35-46`).
Outputs → `notebooks/Figures/other_lms/`.

In [ ]:
import os, sys
_d=os.getcwd()
while _d!='/' and not os.path.isdir(os.path.join(_d,'src')): _d=os.path.dirname(_d)
sys.path.insert(0,_d); os.chdir(_d)
from src.utils import env_utils
sys.path.insert(0, os.path.join(env_utils.PROJECT_ROOT,'notebooks','fig_n_table_generators'))

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap
import figstyle as F; F.set_style()

MODELS = ['Qwen3-32B', 'Olmo-3.1-32B-Instruct']
GROUP = 'other_lms'
CODER = 'Qwen2.5-Coder-32B-Instruct'
COLOR = F.MODEL_COLOR[CODER]          # same colour for every LM (matches the Coder-32B figures)
ROS = lambda m: '' if m == CODER else '__ros-replace-a1-L35-46'
KVR = lambda m: os.path.join(env_utils.DEFAULT_RESULTS_DIR, 'kv_transfer', m, 'task_span')
QR  = lambda m: os.path.join(env_utils.DEFAULT_RESULTS_DIR, 'q_transfer', m)

DTS   = ['int', 'str', 'list[int]', 'list[str]']
LANGS = ['python-python', 'python-cpp', 'cpp-python', 'cpp-cpp']
LANG_LBL = {'python-python': 'py$\\to$py', 'python-cpp': 'py$\\to$cpp', 'cpp-python': 'cpp$\\to$py', 'cpp-cpp': 'cpp$\\to$cpp'}
KV_DT = {'int': 'int', 'str': 'str', 'list[int]': 'list_int', 'list[str]': 'list_str'}
Q_DT  = {'int': 'int', 'str': 'str', 'list[int]': 'list(int)', 'list[str]': 'list(str)'}
VAR_LBL = {'order_matched': 'order-matched', 'order_diff': 'order-diff'}

def _read(p):
    d = json.load(open(p)); return d[0] if isinstance(d, list) else d

def kv_grid(m):
    arm = os.path.join(KVR(m), 'I1-off__I2-off__I3-on-V4__win-span__span-list+gap__cppnote-sys__a1__th-goal_250__gh-goal_250__ro-goal_250' + ROS(m))
    return {dt: {lg: (lambda e: (e['causal_trigger_rate'], e['readout_shift_rate']))(_read(os.path.join(arm, f'{KV_DT[dt]}__{lg}', 'meta.json')))
                 for lg in LANGS} for dt in DTS}

def q_grid(m, v):
    arm = os.path.join(QR(m), f'ph-goal_250__a2__piv-eq__cl-ppgazerestrict-anceq{ROS(m)}__seq')
    return {dt: {lg: (lambda e: (e['causal_trigger_rate'], e['readout_shift_rate']))(_read(os.path.join(arm, v, f'{Q_DT[dt]}__{lg}', 'summary.json')))
                 for lg in LANGS} for dt in DTS}

def mean_over(g, axis):
    cats = LANGS if axis == 'lang' else DTS
    cell = (lambda c: [g[dt][c] for dt in DTS]) if axis == 'lang' else (lambda c: [g[c][lg] for lg in LANGS])
    return cats, np.array([np.mean([x[0] for x in cell(c)]) for c in cats]), np.array([np.mean([x[1] for x in cell(c)]) for c in cats])

def grouped_metric_bars(ax, cats, labels, causal, readout, w=0.38):
    x = np.arange(len(cats))
    ax.bar(x - w/2, causal,  w, **F.metric_kw('causal',  COLOR))
    ax.bar(x + w/2, readout, w, **F.metric_kw('readout', COLOR))
    for cont in ax.containers: ax.bar_label(cont, fmt='%.2f', padding=1.5, fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylim(0, 1.08); ax.set_ylabel('shift rate')

def legend_top(fig, y):
    hs = [Patch(label='causal-shift',  facecolor=COLOR, edgecolor='white', linewidth=0.6),
          Patch(label='readout-shift', facecolor=COLOR, edgecolor='white', linewidth=0.6, hatch=F.HATCH)]
    fig.legend(handles=hs, labels=[h.get_label() for h in hs], loc='lower center', bbox_to_anchor=(0.5, y),
               ncol=2, frameon=False, handlelength=1.6, columnspacing=1.4)

In [ ]:
# ── task span vs code-progress span: one row per LM (residual | KV on 250 goal heads, V×1, 2, 4) ──
def tvp_summary(m):
    root = os.path.join(env_utils.DEFAULT_RESULTS_DIR, 'task_vs_progress', m)
    return {d: json.load(open(os.path.join(root, f'{s}__python-python__seed1123__mx64__ro0__task+gap.summary.json'))) for d, s in KV_DT.items()}

def tvp_grid(T, method, heads=None, v=1.0):
    name = lambda span: 'clean DST (no-op)' if method == 'none' else f"{method}{'·' + heads if heads else ''} · {span} · V×{v:g}"
    r = lambda n: float(np.mean([T[d][n]['causal_shift'] for d in DTS]))
    return np.array([[r('clean DST (no-op)'), r(name('progress'))], [r(name('task')), r(name('task+progress'))]])

CMAP_KV  = LinearSegmentedColormap.from_list('tvp', ['#f3f7fb', COLOR])
CMAP_RES = LinearSegmentedColormap.from_list('tvp_res', ['#fbf3f2', 'firebrick'])
def draw(ax, M, title, cmap, ylab):
    ax.imshow(M, cmap=cmap, vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center', fontsize=10, color='white' if M[i, j] > 0.55 else '#1a1a1a')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1]); ax.set_xticklabels(['DST', 'SRC']); ax.set_yticklabels(['DST', 'SRC'] if ylab else [])
    ax.set_xlabel('code-progress span from', fontsize=9)
    if ylab: ax.set_ylabel('task span from')
    ax.tick_params(which='both', length=0)
    for s in ax.spines.values(): s.set_visible(False)
    ax.set_xticks([0.5], minor=True); ax.set_yticks([0.5], minor=True)
    ax.grid(which='minor', color='white', linewidth=3, alpha=1); ax.grid(which='major', visible=False)
    ax.set_title(title, fontsize=9)

fig = plt.figure(figsize=(10.2, 3.1 * len(MODELS)))
for sf, m in zip(fig.subfigures(len(MODELS), 1, hspace=0.04), MODELS):
    T = tvp_summary(m); sf.suptitle(m, fontsize=11, fontweight='bold')
    axes = sf.subplots(1, 4, gridspec_kw=dict(wspace=0.35))
    draw(axes[0], tvp_grid(T, 'residual'), 'residual transfer', CMAP_RES, True)
    for c, (ax, v) in enumerate(zip(axes[1:], [1.0, 2.0, 4.0])):
        draw(ax, tvp_grid(T, 'kv', 'goal250', v), f'KV, 250 goal heads, V$\\times${v:g}', CMAP_KV, c == 0)
print('saved', F.save_fig(fig, GROUP, 'tvp_grids_by_lm')); plt.show()

In [ ]:
# ── KV transfer of the task span: one row per LM (by language direction | by datatype) ──
fig = plt.figure(figsize=(10, 3.3 * len(MODELS) + 0.4))
top, rows = fig.subfigures(2, 1, height_ratios=[0.4, 3.3 * len(MODELS)])
legend_top(top, 0.0)
for sf, m in zip(rows.subfigures(len(MODELS), 1, hspace=0.04), MODELS):
    sf.suptitle(m, fontsize=11, fontweight='bold')
    axes = sf.subplots(1, 2, sharey=True)
    sf.subplots_adjust(top=0.80, bottom=0.12)
    g = kv_grid(m)
    for ax, (axis, sub) in zip(axes, [('lang', 'by language direction'), ('dt', 'by datatype')]):
        cats, c, r = mean_over(g, axis)
        grouped_metric_bars(ax, cats, [LANG_LBL[x] for x in cats] if axis == 'lang' else cats, c, r)
        ax.set_title(sub, fontsize=10)
    print(m, 'overall causal %.3f readout %.3f' % tuple(np.mean([g[d][l][k] for d in DTS for l in LANGS]) for k in (0, 1)))
print('saved', F.save_fig(fig, GROUP, 'kv_barcharts_by_lm')); plt.show()

In [ ]:
# ── query transfer (gaze-select-restrict): one block per LM, rows = {order-matched, order-diff} ──
fig = plt.figure(figsize=(10, 5.6 * len(MODELS) + 0.4))
top, rows = fig.subfigures(2, 1, height_ratios=[0.4, 5.6 * len(MODELS)])
legend_top(top, 0.0)
for sf, m in zip(rows.subfigures(len(MODELS), 1, hspace=0.04), MODELS):
    sf.suptitle(m, fontsize=11, fontweight='bold')
    axes = sf.subplots(2, 2, sharey=True, gridspec_kw=dict(hspace=0.45))
    for i, v in enumerate(['order_matched', 'order_diff']):
        g = q_grid(m, v)
        for j, (axis, sub) in enumerate([('lang', 'by language direction'), ('dt', 'by datatype')]):
            cats, c, r = mean_over(g, axis)
            grouped_metric_bars(axes[i][j], cats, [LANG_LBL[x] for x in cats] if axis == 'lang' else cats, c, r)
            axes[i][j].set_title(f'{VAR_LBL[v]} · {sub}', fontsize=10)
        print(m, v, 'overall causal %.3f readout %.3f' % tuple(np.mean([g[d][l][k] for d in DTS for l in LANGS]) for k in (0, 1)))
print('saved', F.save_fig(fig, GROUP, 'q_barcharts_by_lm')); plt.show()

## Draft (2026-11-10): Fig 37 with 4 LMs — Devstral-24B, Qwen3-32B, OLMo-3.1-32B-Instruct, GLM-4.5-Air

Same layout as `tvp_grids_by_lm` (needs cell 3's `draw`, `CMAP_*`, `tvp_summary`, `tvp_grid`). Sources:
Qwen3 / OLMo = `results/notebook_caches/task_vs_progress/` (250 goal heads); Devstral = `results/task_vs_progress/` (its 125-head large set; summary keys still say
`goal250`); GLM = the campaign's pooled means in `results/external/glm_devstral_headline.tex`
(387-head large set; per-datatype summaries not on this machine; the no-op cell is 0 by construction).
Saves `tvp_grids_by_lm_4lm` (the paper's `tvp_grids_by_lm` is untouched).

In [ ]:
LM4 = [('Devstral-24B',           'Devstral-Small-2-24B-Instruct-2512', 125),
       ('Qwen3-32B',              'Qwen3-32B',                          250),
       ('OLMo-3.1-32B-Instruct',  'Olmo-3.1-32B-Instruct',              250),
       ('GLM-4.5-Air',            'GLM-4.5-Air',                        387)]
DEV_TVP = os.path.join(env_utils.DEFAULT_RESULTS_DIR, 'task_vs_progress')
# GLM-4.5-Air, pooled over the 4 datatypes (REF/other_models/glm_devstral_headline.txt, section 2)
GLM_TVP = {'residual': {'task': .625, 'progress': .312, 'task+progress': 1.00},
           1.0: {'task': .617, 'progress': .000, 'task+progress': .898},
           2.0: {'task': .758, 'progress': .031, 'task+progress': .820},
           4.0: {'task': .758, 'progress': .281, 'task+progress': .891}}
def grid_from(d): return np.array([[0.0, d['progress']], [d['task'], d['task+progress']]])

def tvp_grids_4lm(slug):
    if slug == 'GLM-4.5-Air':
        return grid_from(GLM_TVP['residual']), [grid_from(GLM_TVP[v]) for v in (1.0, 2.0, 4.0)]
    if slug.startswith('Devstral'):
        T = {d: json.load(open(os.path.join(DEV_TVP, slug, f'{s}__python-python__seed1123__mx64__ro0__task+gap.summary.json')))
             for d, s in KV_DT.items()}
    else:
        T = tvp_summary(slug)
    return tvp_grid(T, 'residual'), [tvp_grid(T, 'kv', 'goal250', v) for v in (1.0, 2.0, 4.0)]

fig = plt.figure(figsize=(10.2, 2.45 * len(LM4)))   # tighter rows than the 2-LM figure
for sf, (name, slug, nh) in zip(fig.subfigures(len(LM4), 1, hspace=0.0), LM4):
    R, KV = tvp_grids_4lm(slug); sf.suptitle(name, fontsize=11, fontweight='bold')
    axes = sf.subplots(1, 4, gridspec_kw=dict(wspace=0.35))
    draw(axes[0], R, 'residual transfer', CMAP_RES, True)
    for c, (ax, v, M) in enumerate(zip(axes[1:], [1.0, 2.0, 4.0], KV)):
        draw(ax, M, f'KV, {nh} goal heads, V$\\times${v:g}', CMAP_KV, c == 0)
        ax.set_ylabel('')   # one shared 'task span from' label, on the residual panel only
    print(f'{name:24s} residual task/prog/both = {R[1,0]:.3f}/{R[0,1]:.3f}/{R[1,1]:.3f}; KV task V×1/2/4 = ' + '/'.join(f'{M[1,0]:.3f}' for M in KV))
print('saved', F.save_fig(fig, GROUP, 'tvp_grids_by_lm_4lm', also_png=False)); plt.show()

## Draft (2026-11-10): Figs 38 / 39 with 4 LMs

GLM-4.5-Air (387 heads) and Devstral-24B (125 heads) grids are parsed from the collaborator's
`results/external/glm_devstral_grids.tex` (16 cells x 32 instances each, generated from the runs'
per-cell `meta.json` / `summary.json`); each parsed grid's mean is checked against the file's stated mean.
Readout-shift for GLM / Devstral = the **fitted** readout constants (α 0.2; GLM layers 21-29, Devstral 18-25);
Qwen3 / OLMo = their replace α 1 readout, as in the paper's figure.
- `kv_barcharts_by_lm_4lm`: Fig 38 layout, one row per LM.
- `q_barcharts_by_lm_4lm`: Fig 39 redesigned: readout-shift dropped; per LM one row of grouped causal-shift bars,
  order-matched vs order-diff, by language direction | by datatype.

In [ ]:
import re
REF_GRIDS = os.path.join(env_utils.DEFAULT_RESULTS_DIR, 'external', 'glm_devstral_grids.tex')   # collaborator's per-cell grids
_TEX = open(REF_GRIDS).read()
_COL = ['python-python', 'python-cpp', 'cpp-python', 'cpp-cpp']       # py->py, py->cpp, cpp->py, cpp->cpp

def ref_grid(title):
    # parse the tabular after the comment line that starts with `title`; returns {dt: {lang: (causal, readout|nan)}}
    i = _TEX.index('% ' + title); block = _TEX[i:_TEX.index('\\end{tabular}', i)]
    stated = float(re.search(r'mean over the 16 cells: ([0-9.]+)', block).group(1))
    g = {}
    for dt, cells in re.findall(r'\\texttt\{([^}]*)\} & (.*?) \\\\', block):
        vals = [c.strip().split(' / ') for c in cells.split('&')]
        g[dt] = {lg: (float(v[0]), float(v[1]) if len(v) > 1 else np.nan) for lg, v in zip(_COL, vals)}
    assert list(g) == DTS, (title, list(g))
    got = np.mean([g[d][l][0] for d in DTS for l in _COL])
    assert abs(got - stated) < 0.006, (title, got, stated)       # cells are printed to 2 decimals
    return g

REF_KV = {'GLM-4.5-Air': ref_grid('GLM-4.5-Air: KV transfer of the task span, V x4, readout with the fitted constants'),
          'Devstral-Small-2-24B-Instruct-2512': ref_grid('Devstral-24B: KV transfer of the task span, V x4, readout with the fitted constants')}
REF_Q = {('GLM-4.5-Air', 'order_matched'): ref_grid('GLM-4.5-Air: Q transfer (GSR), order-matched, readout with the fitted'),
         ('GLM-4.5-Air', 'order_diff'):    ref_grid('GLM-4.5-Air: Q transfer (GSR), order-diff, readout with the fitted'),
         ('Devstral-Small-2-24B-Instruct-2512', 'order_matched'): ref_grid('Devstral-24B: Q transfer (GSR), order-matched, readout with the fitted'),
         ('Devstral-Small-2-24B-Instruct-2512', 'order_diff'):    ref_grid('Devstral-24B: Q transfer (GSR), order-diff, readout with the fitted')}
kv_grid4 = lambda slug: REF_KV[slug] if slug in REF_KV else kv_grid(slug)
q_grid4  = lambda slug, v: REF_Q[(slug, v)] if (slug, v) in REF_Q else q_grid(slug, v)
LBL = lambda axis, cats: [LANG_LBL[x] for x in cats] if axis == 'lang' else cats

# ── Fig 38 (4 LMs): KV transfer of the task span, causal + readout, by language direction | by datatype ──
fig = plt.figure(figsize=(10, 2.9 * len(LM4) + 0.4))
top, rows = fig.subfigures(2, 1, height_ratios=[0.4, 2.9 * len(LM4)])
legend_top(top, 0.0)
for sf, (name, slug, nh) in zip(rows.subfigures(len(LM4), 1, hspace=0.0), LM4):
    sf.suptitle(f'{name} ({nh} goal heads)', fontsize=11, fontweight='bold')
    axes = sf.subplots(1, 2, sharey=True); sf.subplots_adjust(top=0.80, bottom=0.14)
    g = kv_grid4(slug)
    for ax, (axis, sub) in zip(axes, [('lang', 'by language direction'), ('dt', 'by datatype')]):
        cats, c, r = mean_over(g, axis); grouped_metric_bars(ax, cats, LBL(axis, cats), c, r); ax.set_title(sub, fontsize=10)
    print(f'{name:24s} KV overall causal %.3f readout %.3f' % tuple(np.mean([g[d][l][k] for d in DTS for l in LANGS]) for k in (0, 1)))
print('saved', F.save_fig(fig, GROUP, 'kv_barcharts_by_lm_4lm', also_png=False)); plt.show()

In [ ]:
# ── Fig 39 (4 LMs), redesigned: causal-shift only, order-matched (light) vs order-diff (solid) ──
from matplotlib.colors import to_rgb
LIGHT = tuple(0.45 * np.array(to_rgb(COLOR)) + 0.55)   # COLOR mixed 55% with white
def grouped_variant_bars(ax, cats, labels, matched, diff, w=0.38):
    x = np.arange(len(cats))
    ax.bar(x - w/2, matched, w, color=LIGHT, edgecolor='white', linewidth=0.6)
    ax.bar(x + w/2, diff,    w, color=COLOR, edgecolor='white', linewidth=0.6)
    for cont in ax.containers: ax.bar_label(cont, fmt='%.2f', padding=1.5, fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylim(0, 1.08); ax.set_ylabel('causal-shift')

fig = plt.figure(figsize=(10, 2.9 * len(LM4) + 0.4))
top, rows = fig.subfigures(2, 1, height_ratios=[0.4, 2.9 * len(LM4)])
hs = [Patch(label='order-matched', facecolor=LIGHT, edgecolor='white', linewidth=0.6),
      Patch(label='order-diff',    facecolor=COLOR, edgecolor='white', linewidth=0.6)]
top.legend(handles=hs, labels=[h.get_label() for h in hs], loc='lower center', bbox_to_anchor=(0.5, 0.0),
           ncol=2, frameon=False, handlelength=1.6, columnspacing=1.4)
for sf, (name, slug, nh) in zip(rows.subfigures(len(LM4), 1, hspace=0.0), LM4):
    sf.suptitle(f'{name} ({nh} goal heads)', fontsize=11, fontweight='bold')
    axes = sf.subplots(1, 2, sharey=True); sf.subplots_adjust(top=0.80, bottom=0.14)
    gm, gd = q_grid4(slug, 'order_matched'), q_grid4(slug, 'order_diff')
    for ax, (axis, sub) in zip(axes, [('lang', 'by language direction'), ('dt', 'by datatype')]):
        cats, cm, _ = mean_over(gm, axis); _, cd, _ = mean_over(gd, axis)
        grouped_variant_bars(ax, cats, LBL(axis, cats), cm, cd); ax.set_title(sub, fontsize=10)
    print(f'{name:24s} Q causal matched %.3f diff %.3f' % tuple(np.mean([g[d][l][0] for d in DTS for l in LANGS]) for g in (gm, gd)))
print('saved', F.save_fig(fig, GROUP, 'q_barcharts_by_lm_4lm', also_png=False)); plt.show()